# Portfolio Factor Exploration

This notebook generates many factor portfolio combinations across the `all`, `top500`, and `top300` universes, applies transaction-cost scenarios, ranks the results, and plots equity/drawdown charts in the same spirit as the website Analysis tab.

It does not reimplement the backtest math in Python. Python creates the portfolio sweep and plots the results; the actual portfolio returns come from `scripts/notebook-backtest-runner.mjs`, which calls the same `src/server/backtest-engine.js` logic used as the local correctness oracle for the site.


In [ ]:
from pathlib import Path
import itertools
import json
import os
import random
import subprocess
import time

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "scripts" / "notebook-backtest-runner.mjs").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "scripts" / "notebook-backtest-runner.mjs").exists():
    raise FileNotFoundError("Run this notebook from inside factorboosting.github.io.")

os.environ.setdefault("XDG_CACHE_HOME", str(REPO_ROOT / ".cache"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib-cache"))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUTPUT_DIR = REPO_ROOT / "notebooks" / "portfolio_sweep_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#e5e7eb",
    "axes.labelcolor": "#475569",
    "axes.titlecolor": "#0f172a",
    "xtick.color": "#64748b",
    "ytick.color": "#64748b",
    "font.size": 10,
    "axes.grid": True,
    "grid.color": "#f1f5f9",
    "grid.linewidth": 1,
})

print(f"Repo root: {REPO_ROOT}")
print(f"Outputs:   {OUTPUT_DIR}")


In [ ]:
# Sweep controls. Increase the random counts or switch on the pair grid when you want a wider search.
# Full available history starts at 2003-10; the default uses the faster 2016-06 smoke-test window.
START_MONTH = "2003-10"
END_MONTH = "2026-05"
UNIVERSES = ["all", "top500", "top300"]
TRANSACTION_COST_BPS = [0, 50, 100]
ACTIVE_BENCHMARK_ID = "nifty500"

RANDOM_SEED = 42
N_RANDOM_LONG_ONLY = 100
N_RANDOM_LONG_SHORT = 100
MAX_FACTORS_PER_RANDOM_PORTFOLIO = 3

# This can create hundreds of extra base portfolios before universe/cost expansion.
INCLUDE_ALL_SINGLE_FACTOR_LONG_ONLY = True
INCLUDE_ALL_TWO_FACTOR_LONG_ONLY = False

FACTOR_DEFS = {
    "Size": {"labels": {"S": "Small", "B": "Big"}, "preferred": ("S", "B")},
    "Book-to-Market": {"labels": {"V": "Value", "N": "Neutral", "G": "Growth"}, "preferred": ("V", "G")},
    "Profitability": {"labels": {"R": "Robust", "N": "Neutral", "W": "Weak"}, "preferred": ("R", "W")},
    "Investment": {"labels": {"C": "Conservative", "N": "Neutral", "A": "Aggressive"}, "preferred": ("C", "A")},
    "Momentum": {"labels": {"W": "Winner", "N": "Neutral", "L": "Loser"}, "preferred": ("W", "L")},
    "Asset Turnover": {"labels": {"H": "High", "N": "Neutral", "L": "Low"}, "preferred": ("H", "L")},
    "Sales Growth": {"labels": {"H": "High", "N": "Neutral", "L": "Low"}, "preferred": ("H", "L")},
    "Accruals": {"labels": {"C": "Conservative", "N": "Neutral", "A": "Aggressive"}, "preferred": ("C", "A")},
    "Volatility": {"labels": {"L": "Low", "N": "Neutral", "H": "High"}, "preferred": ("L", "H")},
    "Short-Term Reversal": {"labels": {"L": "Loser", "N": "Neutral", "H": "Winner"}, "preferred": ("L", "H")},
}

FACTOR_ORDER = list(FACTOR_DEFS)


In [ ]:
def copy_filters(filters):
    return {factor: list(labels) for factor, labels in (filters or {}).items() if labels}


def filters_to_text(filters):
    filters = copy_filters(filters)
    if not filters:
        return "All"
    pieces = []
    for factor in FACTOR_ORDER:
        if factor in filters:
            pieces.append(f"{factor}={''.join(filters[factor])}")
    for factor in sorted(set(filters) - set(FACTOR_ORDER)):
        pieces.append(f"{factor}={''.join(filters[factor])}")
    return " | ".join(pieces)


def filters_signature(filters):
    return json.dumps(copy_filters(filters), sort_keys=True, separators=(",", ":"))


def factor_tags(long_filters, short_filters=None):
    tags = set(copy_filters(long_filters))
    tags.update(copy_filters(short_filters))
    return sorted(tags, key=lambda item: FACTOR_ORDER.index(item) if item in FACTOR_ORDER else 999)


def make_base_portfolio(name, strategy, long_filters, short_filters=None, source="known"):
    long_filters = copy_filters(long_filters)
    short_filters = copy_filters(short_filters)
    key = "|".join([
        strategy,
        filters_signature(long_filters),
        filters_signature(short_filters),
    ])
    return {
        "baseId": key,
        "name": name,
        "source": source,
        "factorLabel": ", ".join(factor_tags(long_filters, short_filters)) or "All",
        "tags": factor_tags(long_filters, short_filters),
        "config": {
            "strategy": strategy,
            "longFilters": long_filters,
            "shortFilters": short_filters,
        },
    }


def build_known_portfolios():
    p = []

    # Long-only building blocks and familiar tilts.
    p.extend([
        make_base_portfolio("All stocks", "long_only", {}, source="known-long-only"),
        make_base_portfolio("Small caps", "long_only", {"Size": ["S"]}, source="known-long-only"),
        make_base_portfolio("Big caps", "long_only", {"Size": ["B"]}, source="known-long-only"),
        make_base_portfolio("Value", "long_only", {"Book-to-Market": ["V"]}, source="known-long-only"),
        make_base_portfolio("Growth", "long_only", {"Book-to-Market": ["G"]}, source="known-long-only"),
        make_base_portfolio("Momentum winners", "long_only", {"Momentum": ["W"]}, source="known-long-only"),
        make_base_portfolio("Momentum losers", "long_only", {"Momentum": ["L"]}, source="known-long-only"),
        make_base_portfolio("Robust profitability", "long_only", {"Profitability": ["R"]}, source="known-long-only"),
        make_base_portfolio("Weak profitability", "long_only", {"Profitability": ["W"]}, source="known-long-only"),
        make_base_portfolio("Conservative investment", "long_only", {"Investment": ["C"]}, source="known-long-only"),
        make_base_portfolio("Aggressive investment", "long_only", {"Investment": ["A"]}, source="known-long-only"),
        make_base_portfolio("Low volatility", "long_only", {"Volatility": ["L"]}, source="known-long-only"),
        make_base_portfolio("High volatility", "long_only", {"Volatility": ["H"]}, source="known-long-only"),
        make_base_portfolio("Small value", "long_only", {"Size": ["S"], "Book-to-Market": ["V"]}, source="known-combo"),
        make_base_portfolio("Small momentum winners", "long_only", {"Size": ["S"], "Momentum": ["W"]}, source="known-combo"),
        make_base_portfolio("Value momentum", "long_only", {"Book-to-Market": ["V"], "Momentum": ["W"]}, source="known-combo"),
        make_base_portfolio("Quality momentum", "long_only", {"Profitability": ["R"], "Momentum": ["W"]}, source="known-combo"),
        make_base_portfolio("Quality value", "long_only", {"Profitability": ["R"], "Book-to-Market": ["V"]}, source="known-combo"),
        make_base_portfolio("Conservative value", "long_only", {"Investment": ["C"], "Book-to-Market": ["V"]}, source="known-combo"),
        make_base_portfolio("Low-vol quality", "long_only", {"Volatility": ["L"], "Profitability": ["R"]}, source="known-combo"),
        make_base_portfolio("Small quality value", "long_only", {"Size": ["S"], "Profitability": ["R"], "Book-to-Market": ["V"]}, source="known-combo"),
        make_base_portfolio("Small value winners", "long_only", {"Size": ["S"], "Book-to-Market": ["V"], "Momentum": ["W"]}, source="known-combo"),
        make_base_portfolio("Quality value winners", "long_only", {"Profitability": ["R"], "Book-to-Market": ["V"], "Momentum": ["W"]}, source="known-combo"),
        make_base_portfolio("Conservative quality value", "long_only", {"Investment": ["C"], "Profitability": ["R"], "Book-to-Market": ["V"]}, source="known-combo"),
        make_base_portfolio("High asset-turnover sales-growth", "long_only", {"Asset Turnover": ["H"], "Sales Growth": ["H"]}, source="known-combo"),
        make_base_portfolio("Conservative accrual momentum", "long_only", {"Accruals": ["C"], "Momentum": ["W"]}, source="known-combo"),
        make_base_portfolio("Reversal losers plus low vol", "long_only", {"Short-Term Reversal": ["L"], "Volatility": ["L"]}, source="known-combo"),
    ])

    # Classic spread portfolios and economically motivated multi-factor spreads.
    spreads = [
        ("SMB small-minus-big", {"Size": ["S"]}, {"Size": ["B"]}),
        ("HML value-minus-growth", {"Book-to-Market": ["V"]}, {"Book-to-Market": ["G"]}),
        ("MOM winner-minus-loser", {"Momentum": ["W"]}, {"Momentum": ["L"]}),
        ("RMW robust-minus-weak", {"Profitability": ["R"]}, {"Profitability": ["W"]}),
        ("CMA conservative-minus-aggressive", {"Investment": ["C"]}, {"Investment": ["A"]}),
        ("Asset turnover high-minus-low", {"Asset Turnover": ["H"]}, {"Asset Turnover": ["L"]}),
        ("Sales growth high-minus-low", {"Sales Growth": ["H"]}, {"Sales Growth": ["L"]}),
        ("Accrual conservative-minus-aggressive", {"Accruals": ["C"]}, {"Accruals": ["A"]}),
        ("Low-vol-minus-high-vol", {"Volatility": ["L"]}, {"Volatility": ["H"]}),
        ("Short-term reversal loser-minus-winner", {"Short-Term Reversal": ["L"]}, {"Short-Term Reversal": ["H"]}),
        ("Quality momentum spread", {"Profitability": ["R"], "Momentum": ["W"]}, {"Profitability": ["W"], "Momentum": ["L"]}),
        ("Quality value spread", {"Profitability": ["R"], "Book-to-Market": ["V"]}, {"Profitability": ["W"], "Book-to-Market": ["G"]}),
        ("Conservative value spread", {"Investment": ["C"], "Book-to-Market": ["V"]}, {"Investment": ["A"], "Book-to-Market": ["G"]}),
        ("Small value winners vs big growth losers", {"Size": ["S"], "Book-to-Market": ["V"], "Momentum": ["W"]}, {"Size": ["B"], "Book-to-Market": ["G"], "Momentum": ["L"]}),
        ("Low-vol quality spread", {"Volatility": ["L"], "Profitability": ["R"]}, {"Volatility": ["H"], "Profitability": ["W"]}),
        ("Efficient growth spread", {"Asset Turnover": ["H"], "Sales Growth": ["H"]}, {"Asset Turnover": ["L"], "Sales Growth": ["L"]}),
    ]
    for name, long_filters, short_filters in spreads:
        p.append(make_base_portfolio(name, "long_short", long_filters, short_filters, source="known-spread"))
    return p


def build_single_factor_long_only_grid():
    portfolios = []
    for factor, meta in FACTOR_DEFS.items():
        for label, label_name in meta["labels"].items():
            portfolios.append(
                make_base_portfolio(f"{factor}: {label_name}", "long_only", {factor: [label]}, source="single-factor-grid")
            )
    return portfolios


def build_two_factor_long_only_grid():
    portfolios = []
    for f1, f2 in itertools.combinations(FACTOR_ORDER, 2):
        for l1, l2 in itertools.product(FACTOR_DEFS[f1]["labels"], FACTOR_DEFS[f2]["labels"]):
            name = f"{f1} {l1} + {f2} {l2}"
            portfolios.append(make_base_portfolio(name, "long_only", {f1: [l1], f2: [l2]}, source="two-factor-grid"))
    return portfolios


def build_random_long_only(n, rng, max_factors=3):
    portfolios = []
    for idx in range(n):
        k = rng.randint(1, max_factors)
        chosen = rng.sample(FACTOR_ORDER, k)
        filters = {factor: [rng.choice(list(FACTOR_DEFS[factor]["labels"]))] for factor in chosen}
        portfolios.append(
            make_base_portfolio(
                f"Random long-only {idx + 1:03d}: {filters_to_text(filters)}",
                "long_only",
                filters,
                source="random-long-only",
            )
        )
    return portfolios


def build_random_long_short(n, rng, max_factors=3):
    oriented = [factor for factor, meta in FACTOR_DEFS.items() if meta.get("preferred")]
    portfolios = []
    for idx in range(n):
        k = rng.randint(1, max_factors)
        chosen = rng.sample(oriented, k)
        long_filters = {}
        short_filters = {}
        for factor in chosen:
            good, bad = FACTOR_DEFS[factor]["preferred"]
            long_filters[factor] = [good]
            short_filters[factor] = [bad]

        # Sometimes compare the spread inside the same size bucket instead of also making a size spread.
        if "Size" not in chosen and rng.random() < 0.35:
            size_label = rng.choice(["S", "B"])
            long_filters["Size"] = [size_label]
            short_filters["Size"] = [size_label]

        portfolios.append(
            make_base_portfolio(
                f"Random spread {idx + 1:03d}: {filters_to_text(long_filters)} vs {filters_to_text(short_filters)}",
                "long_short",
                long_filters,
                short_filters,
                source="random-long-short",
            )
        )
    return portfolios


def dedupe_base_portfolios(portfolios):
    by_id = {}
    for portfolio in portfolios:
        by_id.setdefault(portfolio["baseId"], portfolio)
    return list(by_id.values())


def build_base_portfolios():
    rng = random.Random(RANDOM_SEED)
    portfolios = []
    portfolios.extend(build_known_portfolios())
    if INCLUDE_ALL_SINGLE_FACTOR_LONG_ONLY:
        portfolios.extend(build_single_factor_long_only_grid())
    if INCLUDE_ALL_TWO_FACTOR_LONG_ONLY:
        portfolios.extend(build_two_factor_long_only_grid())
    portfolios.extend(build_random_long_only(N_RANDOM_LONG_ONLY, rng, MAX_FACTORS_PER_RANDOM_PORTFOLIO))
    portfolios.extend(build_random_long_short(N_RANDOM_LONG_SHORT, rng, MAX_FACTORS_PER_RANDOM_PORTFOLIO))
    return dedupe_base_portfolios(portfolios)


def transaction_cost_payload(bps):
    bps = float(bps)
    if bps <= 0:
        return {"mode": "none", "bps": 0}
    return {"mode": "bps", "bps": bps}


def expand_specs(base_portfolios, universes=UNIVERSES, costs=TRANSACTION_COST_BPS):
    specs = []
    for base_index, base in enumerate(base_portfolios):
        for universe in universes:
            for bps in costs:
                spec_id = f"{base_index:04d}-{universe}-tc{int(bps)}"
                specs.append({
                    **base,
                    "specId": spec_id,
                    "universe": universe,
                    "startMonth": START_MONTH,
                    "endMonth": END_MONTH,
                    "activeBenchmarkId": ACTIVE_BENCHMARK_ID,
                    "transactionCost": transaction_cost_payload(bps),
                })
    return specs


In [ ]:
base_portfolios = build_base_portfolios()
sweep_specs = expand_specs(base_portfolios)

print(f"Base portfolios: {len(base_portfolios):,}")
print(f"Sweep specs:      {len(sweep_specs):,}")
print(f"Universes:        {UNIVERSES}")
print(f"TC bps:           {TRANSACTION_COST_BPS}")

preview = pd.DataFrame([
    {
        "name": p["name"],
        "source": p["source"],
        "strategy": p["config"]["strategy"],
        "long": filters_to_text(p["config"]["longFilters"]),
        "short": filters_to_text(p["config"]["shortFilters"]),
    }
    for p in base_portfolios[:20]
])
display(preview)


In [ ]:
def run_backtest_sweep(specs, include_benchmarks=True):
    runner = REPO_ROOT / "scripts" / "notebook-backtest-runner.mjs"
    payload = {"specs": specs, "includeBenchmarkSeries": include_benchmarks}
    started = time.time()
    completed = subprocess.run(
        ["node", str(runner)],
        cwd=REPO_ROOT,
        input=json.dumps(payload),
        capture_output=True,
        text=True,
    )
    elapsed = time.time() - started
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr[-4000:] or completed.stdout[-4000:])

    response = json.loads(completed.stdout)
    print(f"Backtested {len(response['results']):,} portfolios in {elapsed:,.1f}s.")
    if response.get("errors"):
        print(f"Errors: {len(response['errors'])}")
        display(pd.DataFrame(response["errors"]).head(20))
    return response


def tc_bps(item):
    tc = item.get("transactionCost") or {}
    if tc.get("mode") != "bps":
        return 0.0
    return float(tc.get("bps") or 0)


def summarize_results(results, weight="vw"):
    rows = []
    for item in results:
        metrics = item["results"][f"{weight}_metrics"]
        config = item["config"]
        rows.append({
            "specId": item["specId"],
            "baseId": item.get("baseId"),
            "name": item["name"],
            "source": item.get("source", "custom"),
            "universe": item["universe"],
            "tc_bps": tc_bps(item),
            "weight": weight.upper(),
            "strategy": config.get("strategy"),
            "long": filters_to_text(config.get("longFilters")),
            "short": filters_to_text(config.get("shortFilters")),
            "tags": ", ".join(item.get("tags") or []),
            "avg_turnover_pct": item["results"].get("avgTurnover"),
            "growth_multiple": metrics.get("growth_multiple"),
            "annualized_return": metrics.get("annualized_return"),
            "annualized_volatility": metrics.get("annualized_volatility"),
            "sharpe_ratio": metrics.get("sharpe_ratio"),
            "information_ratio": metrics.get("ir"),
            "max_drawdown": metrics.get("max_drawdown"),
            "months": len(item.get("months") or []),
        })
    return pd.DataFrame(rows)


def save_summary(summary, weight):
    path = OUTPUT_DIR / f"portfolio_sweep_summary_{weight.lower()}.csv"
    summary.to_csv(path, index=False)
    print(f"Saved {path}")


In [ ]:
# Set RUN_FULL_SWEEP to False while editing plotting code; set it True to run the full exploration.
RUN_FULL_SWEEP = True

if RUN_FULL_SWEEP:
    sweep_response = run_backtest_sweep(sweep_specs, include_benchmarks=True)
    results = sweep_response["results"]
    benchmark_lookup = sweep_response["benchmarkSeriesByKey"]
    summary_vw = summarize_results(results, weight="vw")
    summary_ew = summarize_results(results, weight="ew")
    save_summary(summary_vw, "vw")
    save_summary(summary_ew, "ew")

    print("Top VW portfolios by Sharpe:")
    display(
        summary_vw
        .sort_values(["sharpe_ratio", "annualized_return"], ascending=False)
        .head(25)
        [["name", "source", "universe", "tc_bps", "strategy", "annualized_return", "annualized_volatility", "sharpe_ratio", "information_ratio", "max_drawdown", "avg_turnover_pct", "long", "short"]]
    )
else:
    results = []
    benchmark_lookup = {}
    summary_vw = pd.DataFrame()
    summary_ew = pd.DataFrame()


In [ ]:
COLORS = ["#3b82f6", "#10b981", "#f59e0b", "#8b5cf6", "#ec4899", "#0ea5e9", "#84cc16", "#f97316", "#14b8a6", "#64748b"]
BENCH_COLOR = "#ef4444"


def style_axis(ax):
    ax.grid(True, color="#f1f5f9", linewidth=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#e5e7eb")
    ax.spines["bottom"].set_color("#e5e7eb")
    ax.tick_params(colors="#64748b")


def month_dates(months, include_initial=False):
    dates = pd.to_datetime([f"{month}-01" for month in months])
    if include_initial and len(dates):
        dates = pd.DatetimeIndex([dates[0] - pd.DateOffset(months=1)]).append(dates)
    return dates


def results_by_id(results):
    return {item["specId"]: item for item in results}


def select_top_results(summary, results, universe="top500", tc_bps=0, metric="sharpe_ratio", n=5, source=None):
    if summary.empty:
        return []
    frame = summary[(summary["universe"] == universe) & (summary["tc_bps"] == float(tc_bps))].copy()
    if source:
        frame = frame[frame["source"].eq(source)]
    frame = frame.sort_values([metric, "annualized_return"], ascending=False).head(n)
    lookup = results_by_id(results)
    return [lookup[spec_id] for spec_id in frame["specId"] if spec_id in lookup]


def plot_equity_curves(selected, weight="vw", benchmark=True, log_scale=False, title=None, ax=None):
    if not selected:
        raise ValueError("No selected results to plot.")
    if ax is None:
        _, ax = plt.subplots(figsize=(13, 6))

    for idx, item in enumerate(selected):
        months = item["months"]
        dates = month_dates(months, include_initial=True)
        curve = [100] + item["results"][f"{weight}_portfolio"]
        label = f"{item['name']} ({item['universe']}, {tc_bps(item):g} bps)"
        ax.plot(dates, curve, color=COLORS[idx % len(COLORS)], linewidth=2, label=label)

    run_keys = {item["runKey"] for item in selected}
    if benchmark and len(run_keys) == 1:
        bundle = benchmark_lookup.get(next(iter(run_keys)), {})
        bench_id = bundle.get("activeBenchmarkId", ACTIVE_BENCHMARK_ID)
        bench = (bundle.get("benchmarkSeries") or {}).get(bench_id)
        if bench:
            dates = month_dates(bundle["months"], include_initial=True)
            ax.plot(dates, [100] + bench["portfolio"], color=BENCH_COLOR, linewidth=2, linestyle=(0, (6, 3)), label=bench_id.upper())

    if log_scale:
        ax.set_yscale("log")
    ax.set_title(title or f"{weight.upper()} portfolio returns")
    ax.set_ylabel("Portfolio value, start = 100")
    ax.legend(loc="upper left", fontsize=8, frameon=False)
    style_axis(ax)
    return ax


def plot_drawdowns(selected, weight="vw", benchmark=True, title=None, ax=None):
    if not selected:
        raise ValueError("No selected results to plot.")
    if ax is None:
        _, ax = plt.subplots(figsize=(13, 4.5))

    for idx, item in enumerate(selected):
        dates = month_dates(item["months"])
        dd = item["results"][f"{weight}_drawdown"]
        color = COLORS[idx % len(COLORS)]
        ax.plot(dates, dd, color=color, linewidth=1.7, label=item["name"])
        ax.fill_between(dates, dd, 0, color=color, alpha=0.08)

    run_keys = {item["runKey"] for item in selected}
    if benchmark and len(run_keys) == 1:
        bundle = benchmark_lookup.get(next(iter(run_keys)), {})
        bench_id = bundle.get("activeBenchmarkId", ACTIVE_BENCHMARK_ID)
        bench = (bundle.get("benchmarkSeries") or {}).get(bench_id)
        if bench:
            dates = month_dates(bundle["months"])
            ax.plot(dates, bench["drawdown"], color=BENCH_COLOR, linewidth=1.7, linestyle=(0, (6, 3)), label=bench_id.upper())

    ax.set_title(title or f"{weight.upper()} drawdown")
    ax.set_ylabel("Drawdown (%)")
    ax.legend(loc="lower left", fontsize=8, frameon=False)
    style_axis(ax)
    return ax


def plot_leaderboard(summary, metric="sharpe_ratio", n=20, universe=None, tc_bps_value=None, title=None):
    frame = summary.copy()
    if universe is not None:
        frame = frame[frame["universe"] == universe]
    if tc_bps_value is not None:
        frame = frame[frame["tc_bps"] == float(tc_bps_value)]
    frame = frame.sort_values([metric, "annualized_return"], ascending=False).head(n)
    labels = frame["name"] + " | " + frame["universe"] + " | " + frame["tc_bps"].astype(int).astype(str) + " bps"

    _, ax = plt.subplots(figsize=(11, max(5, n * 0.32)))
    ax.barh(labels[::-1], frame[metric].iloc[::-1], color="#3b82f6", alpha=0.88)
    ax.set_title(title or f"Top {n} by {metric}")
    ax.set_xlabel(metric)
    style_axis(ax)
    return ax


def plot_tc_sensitivity(summary, metric="sharpe_ratio", universe="top500", top_n=8):
    base = summary[(summary["universe"] == universe) & (summary["tc_bps"] == 0)].copy()
    names = base.sort_values([metric, "annualized_return"], ascending=False).head(top_n)["name"].tolist()
    frame = summary[(summary["universe"] == universe) & (summary["name"].isin(names))]
    pivot = frame.pivot_table(index="tc_bps", columns="name", values=metric, aggfunc="mean").sort_index()

    _, ax = plt.subplots(figsize=(12, 5.5))
    for idx, name in enumerate(pivot.columns):
        ax.plot(pivot.index, pivot[name], marker="o", linewidth=2, color=COLORS[idx % len(COLORS)], label=name)
    ax.set_title(f"Transaction-cost sensitivity ({universe}, {metric})")
    ax.set_xlabel("Transaction cost, bps on traded portfolio value")
    ax.set_ylabel(metric)
    ax.legend(loc="best", fontsize=8, frameon=False)
    style_axis(ax)
    return ax


def plot_factor_heatmap(summary, metric="sharpe_ratio", tc_bps_value=0):
    rows = []
    frame = summary[summary["tc_bps"] == float(tc_bps_value)]
    for _, row in frame.iterrows():
        tags = [tag.strip() for tag in str(row["tags"]).split(",") if tag.strip()]
        for tag in tags or ["All"]:
            rows.append({"factor": tag, "universe": row["universe"], metric: row[metric]})
    heat = pd.DataFrame(rows).pivot_table(index="factor", columns="universe", values=metric, aggfunc="mean")
    heat = heat.reindex([factor for factor in FACTOR_ORDER if factor in heat.index] + [idx for idx in heat.index if idx not in FACTOR_ORDER])

    _, ax = plt.subplots(figsize=(7, max(4, len(heat) * 0.42)))
    image = ax.imshow(heat.values, aspect="auto", cmap="RdYlGn")
    ax.set_xticks(range(len(heat.columns)), heat.columns)
    ax.set_yticks(range(len(heat.index)), heat.index)
    ax.set_title(f"Average {metric} by factor tag ({tc_bps_value:g} bps)")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            value = heat.values[i, j]
            if np.isfinite(value):
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8, color="#0f172a")
    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    return ax


In [ ]:
if not summary_vw.empty:
    top_top500_no_tc = select_top_results(summary_vw, results, universe="top500", tc_bps=0, metric="sharpe_ratio", n=5)
    plot_equity_curves(top_top500_no_tc, weight="vw", benchmark=True, log_scale=False, title="Top Top 500 portfolios, no transaction cost")
    plt.show()

    plot_drawdowns(top_top500_no_tc, weight="vw", benchmark=True, title="Drawdowns for top Top 500 portfolios")
    plt.show()

    plot_leaderboard(summary_vw, metric="sharpe_ratio", n=20, title="VW Sharpe leaderboard across universes and transaction costs")
    plt.show()

    plot_tc_sensitivity(summary_vw, metric="sharpe_ratio", universe="top500", top_n=8)
    plt.show()

    plot_factor_heatmap(summary_vw, metric="sharpe_ratio", tc_bps_value=0)
    plt.show()


In [ ]:
if not summary_vw.empty:
    # Best portfolio curves by universe after 100 bps transaction costs.
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
    for ax, universe in zip(axes, UNIVERSES):
        selected = select_top_results(summary_vw, results, universe=universe, tc_bps=100, metric="sharpe_ratio", n=3)
        plot_equity_curves(selected, weight="vw", benchmark=True, title=f"{universe}: top VW curves after 100 bps", ax=ax)
    plt.tight_layout()
    plt.show()

    # Compare EW and VW rankings.
    comparison = (
        summary_vw[["specId", "name", "universe", "tc_bps", "sharpe_ratio", "annualized_return", "max_drawdown"]]
        .rename(columns={
            "sharpe_ratio": "vw_sharpe",
            "annualized_return": "vw_ann_return",
            "max_drawdown": "vw_max_dd",
        })
        .merge(
            summary_ew[["specId", "sharpe_ratio", "annualized_return", "max_drawdown"]]
            .rename(columns={
                "sharpe_ratio": "ew_sharpe",
                "annualized_return": "ew_ann_return",
                "max_drawdown": "ew_max_dd",
            }),
            on="specId",
            how="left",
        )
        .sort_values("vw_sharpe", ascending=False)
    )
    display(comparison.head(25))
    comparison.to_csv(OUTPUT_DIR / "portfolio_sweep_ew_vw_comparison.csv", index=False)


## Ideas to Try Next

- Set `INCLUDE_ALL_TWO_FACTOR_LONG_ONLY = True` for a broader two-factor grid.
- Increase `N_RANDOM_LONG_ONLY` and `N_RANDOM_LONG_SHORT` for a wider random search.
- Change `TRANSACTION_COST_BPS` to include 10, 150, or 200 bps if you want a finer implementation-cost stress test.
- Change `START_MONTH` to `2003-10` when you want the full available backtest history.
